**Lecture 6 — Class Exercise**

**Part-to-Whole: Hierarchical Visualization**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


**Task 1 — Treemap: fossil fuel dependency by country**

**What to build:** A treemap showing fossil fuel TWh only, broken down by Region → Country → Source (Coal / Oil / Natural Gas).


**Requirements:**


Filter to fossil sources only before plotting
Use path=['Region', 'Country', 'Source'] for the hierarchy
Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
Show TWh values in labels — no percentages
Grey out parent nodes (Region and Country level)
Insight title naming which region or country is most fossil-dependent


💡 df.loc[df['Source_Type'] == 'Fossil']

In [3]:
# Filter to fossil fuels only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD‑safe palette for fossil sources
fossil_colors = {
    "Coal": "#636EFA",         # blue
    "Oil": "#EF553B",          # orange-red
    "Natural Gas": "#00CC96"   # green
}

fig = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],
    values="TWh",                     # ensure this matches your column name
    color="Source",
    color_discrete_map=fossil_colors
)

# Grey out parent nodes (Region + Country)
fig.update_traces(
    marker=dict(
        colorscale=None,
        line=dict(color="white", width=1)
    ),
    tiling=dict(pad=2),
    root_color="lightgrey"
)

# Improve labels: show TWh explicitly
fig.update_traces(
    texttemplate="%{label}<br>%{value} TWh",
    hovertemplate="<b>%{label}</b><br>TWh: %{value}<extra></extra>"
)

# Insight title — adjust once you inspect the data
fig.update_layout(
    title="Insight: Asia shows the highest fossil fuel dependence, driven mainly by China’s coal consumption",
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()


**Task 2 — Sunburst: tipping behaviour by day and meal time**

**What to build:** A sunburst chart using the built-in tips dataset showing how total bill amount is distributed across day → time → smoker status.



**Requirements:**



Load tips with px.data.tips()
Aggregate total bill (sum of total_bill) per group — not count
Hierarchy: path=['day', 'time', 'smoker']
Colour encodes smoker status with a CVD-safe blue/orange palette
Grey out parent nodes (day and time level)
Use percent parent for text labels
Insight title describing where the most spending happens


💡 tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

In [4]:
import plotly.express as px

# Load dataset
tips = px.data.tips()

# Aggregate total bill by day → time → smoker
tips_sum = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
        .sum()
        .reset_index()
)

# CVD‑safe palette for smoker status
smoker_colors = {
    "Yes": "#EF553B",   # orange-red
    "No": "#636EFA"     # blue
}

fig = px.sunburst(
    tips_sum,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_colors
)

# Grey out parent nodes (day + time)
fig.update_traces(
    branchvalues="total",
    insidetextorientation='radial',
    textinfo='label+percent parent',
    hovertemplate="<b>%{label}</b><br>Total Bill: %{value:.2f}<extra></extra>",
    marker=dict(line=dict(color="white", width=1)),
    root_color="lightgrey"
)

fig.update_layout(
    title="Insight: Weekend dinners account for the highest total spending, especially among non‑smokers",
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()


**Task 3 — Treemap vs bar: low-carbon energy by country**

**What to build:** Build both a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**

Filter to Source_Type == 'Low-carbon' and aggregate TWh by country
Treemap: single-level path=['All', 'Country'] with a dummy root node labelled 'Low-carbon'
Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
Both charts show TWh values, not percentages
Insight title on the bar chart naming the leading country

In [8]:
import plotly.express as px

# ----------------------------------------------------
# 1. Filter to low‑carbon sources + aggregate by country
# ----------------------------------------------------

df_lc = df.loc[df['Source_Type'] == 'Low-carbon'].copy()

lc_country = (
    df_lc.groupby('Country', as_index=False)['TWh']
         .sum()
         .sort_values('TWh', ascending=False)
)

# Dummy root node for treemap
lc_country['All'] = 'Low-carbon'


# ----------------------------------------------------
# 2. Treemap (single-level, dummy root)
# ----------------------------------------------------

fig_treemap = px.treemap(
    lc_country,
    path=['All', 'Country'],
    values='TWh',
    color='Country'
)

fig_treemap.update_traces(
    root_color="lightgrey",
    texttemplate="%{label}<br>%{value} TWh",
    hovertemplate="<b>%{label}</b><br>TWh: %{value}<extra></extra>"
)

fig_treemap.update_layout(
    title="Low‑carbon Energy by Country (Treemap)",
    margin=dict(t=60, l=10, r=10, b=10)
)

fig_treemap.show()


# ----------------------------------------------------
# 3. Horizontal Bar Chart (sorted, CVD‑safe)
# ----------------------------------------------------

bar_color = "#1f77b4"   # CVD‑safe blue

fig_bar = px.bar(
    lc_country,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=[bar_color]
)

fig_bar.update_traces(
    texttemplate="%{x} TWh",
    textposition="outside"
)

fig_bar.update_layout(
    title=f"Insight: {lc_country.iloc[0]['Country']} leads in low‑carbon energy production",
    xaxis_title="TWh",
    yaxis_title="Country",
    margin=dict(t=60, l=10, r=10, b=10)
)

fig_bar.show()
